# MNT02 학생용 AI 개발 실습

핵심 구현 → 첫 실행 → 실제 AI와 추가 변경 → 재실행·검토 → 본인 판단을 기록합니다. 교육용 가상 자료이며 실제 부대·장비 데이터를 입력하지 않습니다.


최종본 편성: 28.5시간. 실제소요/승인값이아닙니다. 현재첨부와함께사용합니다.


In [ ]:
from pathlib import Path
import zipfile,sys
try:
 from google.colab import files
 uploaded=files.upload();archive='MNT02_실습프로젝트.zip'
 if archive not in uploaded:raise ValueError('MNT02_실습프로젝트.zip을 선택하세요')
 dest=Path('/content').resolve()
 if (dest/'MNT02_project').exists():print('기존 수정 코드·기록을 재사용합니다. 덮어쓰지 않습니다.')
 else:
  with zipfile.ZipFile(archive) as z:
   for name in z.namelist():
    target=(dest/name).resolve()
    if not target.is_relative_to(dest) or Path(name).parts[0]!='MNT02_project':raise ValueError('실습 ZIP 경로 오류')
   z.extractall(dest)
 ROOT=dest/'MNT02_project'
except ImportError:
 ROOT=Path.cwd() if (Path.cwd()/'pipeline/run_all.py').exists() else Path.cwd()/'MNT02_project'
sys.path.insert(0,str(ROOT));print('프로젝트:',ROOT)
from pipeline.run_all import run_mission


라이브러리가 없을 때만 아래 주석을 풀어 설치 후 런타임을 다시 시작합니다.

In [ ]:
# 필요할 때만 실행
# %pip install numpy pandas scipy scikit-learn matplotlib

In [ ]:
from datetime import datetime
import shutil
PATHS=['src/domain/mnt02_domain.py', 'src/etl/mnt02_data_cleansing.py', 'src/analysis/mnt02_eda.py', 'src/features/mnt02_feature_engineering.py', 'src/models/mnt02_advanced_model.py', 'src/explainability/mnt02_xai_explainer.py', 'src/optimization/mnt02_simulator.py', 'src/dashboard/mnt02_app.py', 'src/analysis/mnt02_aar.py', 'pipeline/mnt02_retrain.py']
mission_number=1
source=ROOT/PATHS[mission_number-1]
backup=ROOT/'backups'/(datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'_'+source.name)
backup.parent.mkdir(exist_ok=True);shutil.copy2(source,backup)
print('수정 전 사본:',backup)
# 검토한 전체 코드를 해당 source파일에 저장한 뒤 다시 실행합니다.

In [ ]:
from tools.patch_function import replace_function
import json
TASKS=json.loads((ROOT/"implementation_tasks.json").read_text())
SELECTED_MISSIONS=list(range(1,11)) # 선택 M4·M6·M8을 제외할 경우 사유 기록


## M01 야전 정비창 수리부속 청구/출고 이력 및 장비 가동률 데이터셋 스키마 정의

AI와 eol_notice_date 필드·불가능 날짜·NaN/무한대·자료형 오류를 검증기에 추가하고 정상/주입 오류 결과를 비교한다.

가상 NSN 형식을 유지하고 실제 코드 의미나 업체를 새로 조사해 넣지 않습니다. 정상 20건 과 오류 주입의 대상별 검출을 구분합니다. 단종 공지일을 추가하면 모든 정상 샘플도 갱신해야 합니다.

forms/MNT02_M01_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

가상 청구 기록 rec 사전 → 오류 문자열 목록. 제공 SCHEMA와 check를 사용해 필수 필드·자료형·허용값을 검사한다. 정상/오류 주입 결과를 구분한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def validate(rec):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/domain/mnt02_domain.py','validate',implementation)


In [ ]:
result=run_mission(1)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M02 결측치 보정, 계절성 수요 분리 및 수리부속 품목코드(NSN) 정제 전처리 모듈 구현

AI와 미등록 NSN·미래 날짜 격리 원문, 규칙별 처리 근거, 정제 후 논리 재검증을 확인하고 단위 환산 변경 결과를 비교한다.

음수·오타·상자 단위 보정은 교육용 주입 원인 전제입니다. 원문·격리 기록을 보존하며 긴급 불출을 단순 이상치로 삭제하지 않습니다. 선형 보간·중심 이동평균은 사후 분석용이며 미래를 쓰지 않는 예측 피처와 구분합니다.

forms/MNT02_M02_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

NSN 원문 s → 가상 표준 NSN 문자열 또는 None. 숫자13자리를 제공 형식으로 정규화하고 원문·격리 기록은 유지한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def fmt_nsn(s):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/etl/mnt02_data_cleansing.py','fmt_nsn',implementation)


In [ ]:
result=run_mission(2)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M03 이동평균 및 지수평활법 기반 1차 부품 소요량 예측 Baseline 알고리즘 구현

SES alpha를 0.6으로 바꾼 결과와 최초 후보 선택 결과를 비교하고 차트 8종·상관검정의 해석을 검증한다.

MAPE는 실제수요 0을 제외한 분모, WAPE는 실제 합계가 0일 때N/A를 표시합니다. 상관과p-value는 인과가 아닙니다. 가상 생성 구조·반복 품목/월·다중 비교·상관된 피처의 한계를 설명합니다.

forms/MNT02_M03_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

수요 배열 y,alpha → 한달 앞 SES 예측 배열. 초기값은 첫 관측값이고 이후 예측은 이전 시점의 관측만 사용한다. 빈/비유한/음수 수요와 범위 밖 alpha는 거부한다. 초기 구간은 성능평가에서 제외한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def ses(y, alpha):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/analysis/mnt02_eda.py','ses',implementation)


In [ ]:
result=run_mission(3)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M04 시계열 머신러닝(Prophet/LSTM) 기반 결전 대비 야전 부품 수요 예측 모델 고도화

ma6·trend_t·lag2 제외 전후 교차검증을 비교하고 예측 시점에 알려진 계획 정보와 실측 정보를 구분한다.

Prophet/LSTM이라는 원본 제목을 유지하되 실제 구현은 원본대로 포아송 HistGradientBoosting입니다. 월별 한달 앞 rolling 평가와 6개 월 고정 기점 예측을 구분합니다. 당월 운용시간은 사전에 알려진 계획값이라는 가정이 필요하고 실제 당월 실적이면 누수입니다.

forms/MNT02_M04_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

hist DataFrame → 품목별 과거 수요와 제공된 계획 변수로 만든 파생 피처 DataFrame. lag/이동평균/계절지수는 미래 수요를 참조하지 않는다. 당월 운용시간은 사전 계획값이라는 가정을 명시한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def make_features(hist):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/features/mnt02_feature_engineering.py','make_features',implementation)


In [ ]:
result=run_mission(4)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M05 적정 안전재고량 산출 및 결품 위험 지수(Stockout Risk) 엔진 개발

단종예고 가중치 0.5→0.8 변경 전후의 위험 품목 수·점수 변화를 비교하고 모델 성능과 위험 지수를 분리한다.

가중치는t36~41에서 고르고t42~47을 성능 평가에 사용합니다. 다음달 위험 계산에는 당시까지 알려진 과거 오차를 사용할 수 있습니다. 95%z는 분포 가정이며 실측 충족률 보장이 아닙니다. 위험 지수 0~100은 보정된 단종 확률이 아닙니다.

forms/MNT02_M05_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

items,mu,sigma,z,review → 품목별 mu/sigma/L/ss/rop DataFrame. 제공 리드타임+검토주기로 안전재고·재주문점을 계산한다. 정규분포 가정이며 실제 서비스 수준을 보장하지 않는다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def stock_policy(items, mu, sigma, z=1.65, review=1):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/models/mnt02_advanced_model.py','stock_policy',implementation)


In [ ]:
result=run_mission(5)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M06 야전 부품 재고 현황 및 긴급 조달 필요 품목 가시화 대시보드 구현

위험 60/70 기준의 목록과 설명을 비교하며 요인별 일회 대조 결과가 가법적 설명인지·인과 주장인지 검토한다.

요인 1개 씩 기준값으로 대조한 변화는 개별 모델 민감도입니다. 합이 원점수와 같거나 요인의 실제 인과 기여라고 보장하지 않습니다. 상위 3개 산출률과 설명의 정확성을 따로 확인합니다. 대시보드는정적 그림이며실시간 웹서비스가 아닙니다.

forms/MNT02_M06_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

iid → 위험 지수와 상위3개 모델 민감도 요인을 설명하는 문자열. 제공 dash/top3/C/CAUSE를 사용하며 실제 고장 원인·인과를 확정하지 않는다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def explain(iid):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/explainability/mnt02_xai_explainer.py','explain',implementation)


In [ ]:
result=run_mission(6)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M07 부품 발주량 최종 확정을 위한 정비장교 HITL 승인/조정 워크플로우 설계

예산 비율 변경의 추천량을 비교하고 별도 직접 입력 도구로 3건 이상 승인·조정·반려/보류를 사유와 함께 기록한다.

원본 코드의 결재는SIMULATED 예시입니다. 직접 입력한 LEARNER_INPUT 로그는별도입니다. 대체품 가용과 호환성 승인은 다르며 자동 집행은 없습니다. 안전재고 보정은평가 전t30~35에서만 구합니다. 해시 체인은내부 변경 탐지이며전면 재작성·끝부분 삭제 방지에는외부 신뢰 기준점이 필요합니다.

forms/MNT02_M07_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

시뮬레이션 결과 r,수요 dem → 가상 비용·충족률·회전율 사전. 수요/재고 분모0은 NaN으로 산출불가를 표시하고 실제 재정효과와 구분한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def summary(r, dem):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/optimization/mnt02_simulator.py','summary',implementation)


In [ ]:
result=run_mission(7)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

### 직접 입력 승인〔가상 역할〕

아래 셀을반복실행해3건 이상 승인·조정·반려/보류를 직접입력합니다. 실제인증·실제발주는아닙니다. LTB 예산은원본미제공이므로기본None이며승인할수없습니다. 별도교육용예산을가정하면근거를forms에기록합니다. 호환성필요후보는근거확인전승인하지않습니다. 기존로그·신뢰기준점은보존합니다.

In [ ]:
import json
from src.approval_review import ReviewFlow
record=ROOT/'outputs/learner_audit.json';record.parent.mkdir(exist_ok=True)
candidates=json.loads((ROOT/'data/review_candidates.json').read_text());budgets=json.loads((ROOT/'data/review_budgets.json').read_text())
f=ReviewFlow(candidates,budgets)
if record.exists():
 saved=json.loads(record.read_text());f.docs=saved['docs'];f.budgets=saved['budgets'];f.log=saved['log']
 if saved['source']!='LEARNER_INPUT' or not f.verify(saved['anchor']):raise ValueError('출처·로그 무결성 확인 실패')
print('후보:',f.docs,'가상 예산:',f.budgets)
did=input('후보ID〔Q01 등〕: ').strip();decision=input('승인 / 조정 / 반려 / 보류: ').strip();reason=input('본인 확인 근거와사유: ')
qty=int(input('조정 확정량: ')) if decision=='조정' else None
compat=input('호환성근거를 직접확인했다면 YES〔그외 빈칸〕: ').strip()=='YES'
result=f.decide('maintenance_officer',did,decision,reason,qty,compat)
record.write_text(json.dumps(f.export(),ensure_ascii=False,indent=2));print(result,'저장:',record,'실제 집행 없음')


## M08 전시 작전 템포 증가 모의 부품 수요 300% 폭증 시뮬레이션 부하 테스트

품목군 배수를 변경해 전체 정확도·탐지율·경계 제외 비율을 비교하고 50배 데이터 갱신 시간을 측정한다.

300%는 평시의 3배입니다. 전체 정확도가 95% 미달하면그대로 기록하고 경계 구간 제외 정확도·제외 비율·남은 분모를 별도 표시합니다. 경계 제외로전체 목표 PASS를 만들지 않습니다. 50배는기존 품목 복제 부하이며 2,000개 독립 품목의 정확도 검증이 아닙니다.

forms/MNT02_M08_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

Xb,it,st → (품목군별 위험 집계,품목별 위험표). 제공 ens/SURGE와 정책·위험 함수를 연결한다. 50배는 복제 품목 부하이며 실제 독립 품목 정확도와 웹 화면 갱신 시간이 아니다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def refresh(Xb, it, st):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/dashboard/mnt02_app.py','refresh',implementation)


In [ ]:
result=run_mission(8)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M09 AAR 기반 부품 과다/과소 재고 예측 오차 분석 및 안전재고 파라미터 재보정

앞 6개 월만으로 ABC·편향·z를 정했는지 검사하고 결품 손실 가정 변경의 뒤 6개 월 결과를 비교한다.

앞 6개 월t36~41에서 ABC·편향·z를 결정하고뒤t42~47에서 검증합니다. 원본의ABC 전체 12개 월 사용과미래 예측 평균을사용한sigma를 보완했습니다. 단가·결품 손실·대기시간은가상 가정이고 실제 예산·가동률 개선 효과가 아닙니다.

forms/MNT02_M09_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

fc,z → (뒤6개월 검증 시뮬레이션 결과,가상 총비용). 제공 D/A/sig/items로 앞6개월에서 정한 파라미터를 평가하며 검증 자료로 다시 보정하지 않는다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def ev(fc, z):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'src/analysis/mnt02_aar.py','ev',implementation)


In [ ]:
result=run_mission(9)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## M10 AI 기반 야전 정비 부품 수요예측 및 재고최적화 시스템 통합 패키지 완성

LTB_MONTHS24→12 변경을 비교하고 선택된 모델·대기열·직접 작성 10장 문서·브리핑 수치를 대조한다.

M10의t42~47은모델 선택 구간이며독립 최종 평가가 아닙니다. 원본에서는v1 유지 후전체 학습 모델로예측하는 불일치가 있어실제 선택v1/v2로예측하도록 고쳤습니다. 가상 2026-01 실제 수요는미제공입니다. 가이드 10문장 출력은 10장 본문 작성이 아니며 시작문서는FAIL입니다. 실제 문서 작성·직접 승인·교관 검토는별도입니다.

forms/MNT02_M10_forms.md에 실제 대화·실행·판단을 기록합니다.

### 구현 함수 저장

운영모델 오차e1,후보모델 오차e2 → v1/v2 문자열. 유한한0이상 오차만 비교하고 e2<=e1이면 v2를 선택한다. 빈/NaN 평가를 거부하고 실제 선택 모델로 이후 예측한다.

전체 def를 작성해 아래 implementation 문자열에 넣습니다. 이름·인자·기본값은 유지합니다. 이전 코드는 .before 파일로 보존합니다.


In [ ]:
# implementation = """
# def choose_version(e1,e2):
#     여기에 실제 구현 작성
# """
# replace_function(ROOT/'pipeline/mnt02_retrain.py','choose_version',implementation)


In [ ]:
result=run_mission(10)
print(result)
from IPython.display import display,Image
for figure in result['figures']:display(Image(filename=str(ROOT/figure)))

## 경계 테스트

통과는 실제 보고서·원본 정량 목표 달성을 대신하지 않습니다.

In [ ]:
import subprocess
p=subprocess.run([sys.executable,str(ROOT/'tests/test_boundaries.py')],capture_output=True,text=True)
print(p.stdout,p.stderr)
if p.returncode:raise RuntimeError('경계 테스트 실패')

## 제출ZIP 만들기

본문·AI대화·직접 승인·수정 코드를 저장한 뒤 실행합니다. 파일 존재 검사이며 평가 PASS를 뜻하지 않습니다.

In [ ]:
import ast,json
from tools.patch_function import find_function
if not {1,2,3,5,7,9,10}.issubset(SELECTED_MISSIONS):raise RuntimeError('원본 필수7미션을 포함해야 합니다')
for task in TASKS:
 if task['mission'] not in SELECTED_MISSIONS:continue
 node=find_function((ROOT/task['path']).read_text(),task['functions'][0])
 if any(isinstance(n,ast.Raise) and isinstance(n.exc,ast.Call) and isinstance(n.exc.func,ast.Name) and n.exc.func.id=='NotImplementedError' for n in ast.walk(node)):raise RuntimeError('미구현 함수: '+task['functions'][0])
 if not list((ROOT/'outputs'/f"M{task['mission']:02d}").glob('*/run.json')):raise RuntimeError('실행 결과 누락: '+str(task['mission']))
from src.approval_review import ReviewFlow
record=ROOT/'outputs/learner_audit.json'
if not record.exists():raise RuntimeError('M7 직접 정비장교 검토 기록이 필요합니다')
saved=json.loads(record.read_text())
f=ReviewFlow([],saved.get('budgets',{}));f.log=saved.get('log',[])
if saved.get('source')!='LEARNER_INPUT' or len(f.log)<3 or not f.verify(saved.get('anchor')) or any(r.get('source')!='LEARNER_INPUT' for r in f.log):raise RuntimeError('직접 검토3건·출처·감사로그 무결성을 확인하세요')
import zipfile
required=['docs/mnt02_interface_spec.md', 'docs/mnt02_logistics_domain_spec.md', 'docs/mnt02_operations_guide.md', 'notebooks/mnt02_eda_analysis.ipynb', 'outputs/learner_audit.json', 'pipeline/mnt02_retrain.py', 'reports/mnt02_briefing.md', 'reports/mnt02_cleansing_quality.md', 'reports/mnt02_data_dictionary.md', 'reports/mnt02_eda_summary.md', 'reports/mnt02_feature_validation.md', 'reports/mnt02_hitl_review.md', 'reports/mnt02_model_metrics.md', 'reports/mnt02_roi_analysis.md', 'reports/mnt02_stress_test.md', 'reports/mnt02_xai_report.md', 'src/dashboard/mnt02_app.py', 'src/etl/mnt02_data_cleansing.py', 'src/explainability/mnt02_xai_explainer.py', 'src/features/mnt02_feature_engineering.py', 'src/models/mnt02_advanced_model.py', 'src/optimization/mnt02_simulator.py', 'forms/MNT02_M01_forms.md', 'forms/MNT02_M02_forms.md', 'forms/MNT02_M03_forms.md', 'forms/MNT02_M04_forms.md', 'forms/MNT02_M05_forms.md', 'forms/MNT02_M06_forms.md', 'forms/MNT02_M07_forms.md', 'forms/MNT02_M08_forms.md', 'forms/MNT02_M09_forms.md', 'forms/MNT02_M10_forms.md']
missing=[x for x in required if not (ROOT/x).exists()]
if missing:raise FileNotFoundError(str(missing))
out=ROOT.parent/'uploads';out.mkdir(exist_ok=True);archive=out/'MNT02.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in ROOT.rglob('*'):
  if p.is_file() and '__pycache__' not in p.parts:z.write(p,p.relative_to(ROOT))
print('학습자 제출 ZIP:',archive)
try:
 from google.colab import files
 files.download(str(archive))
except ImportError:pass